In [ ]:
import os, glob, json, time, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("Keras:", keras.__version__)


In [ ]:
PAMAP_ROOT = r"E:\WISDM_ar_v1.1\PAMAP2_Dataset"

RUN_NAME = "PAMAP2_TRANSFORMER_STREAMLINED"
OUT_DIR = os.path.join(PAMAP_ROOT, "OUTPUT_PAMAP2_TRANSFORMER", RUN_NAME)

DIRS = {
    "conf_mats": os.path.join(OUT_DIR, "confusion_matrices"),
    "models":    os.path.join(OUT_DIR, "models"),
    "plots":     os.path.join(OUT_DIR, "plots"),
    "reports":   os.path.join(OUT_DIR, "reports"),
    "tflite":    os.path.join(OUT_DIR, "tflite"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("OUT_DIR:", OUT_DIR)


In [ ]:
dat_files = sorted(glob.glob(os.path.join(PAMAP_ROOT, "**", "*.dat"), recursive=True))
print("Found .dat files:", len(dat_files))
for f in dat_files[:5]:
    print(" -", f)

In [ ]:
FEATURE_IDXS = [2,4,5,6,10,11,12,21,22,23,27,28,29,38,39,40,44,45,46]
FEATURE_NAMES = [
 "hr",
 "hand_acc_x","hand_acc_y","hand_acc_z",
 "hand_gyro_x","hand_gyro_y","hand_gyro_z",
 "chest_acc_x","chest_acc_y","chest_acc_z",
 "chest_gyro_x","chest_gyro_y","chest_gyro_z",
 "ankle_acc_x","ankle_acc_y","ankle_acc_z",
 "ankle_gyro_x","ankle_gyro_y","ankle_gyro_z"
]


In [ ]:
def load_pamap2(files):
    dfs = []
    for f in files:
        df = pd.read_csv(f, sep=r"\s+", header=None)
        keep = [1] + FEATURE_IDXS  # col 1 = activity_id
        sub = df.iloc[:, keep].copy()
        sub.columns = ["activity_id"] + FEATURE_NAMES
        dfs.append(sub)
    return pd.concat(dfs, ignore_index=True)

data = load_pamap2(dat_files)
print("Raw shape:", data.shape)

# Remove activity 0 (unknown)
data = data[data.activity_id != 0]

# Clean NaN/Inf
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

activity_ids = sorted(data.activity_id.unique())
print("Clean shape:", data.shape)
print("Activities:", activity_ids)


In [ ]:
WIN  = 200
STEP = 100

id2idx = {a:i for i,a in enumerate(activity_ids)}
idx2id = {i:a for a,i in id2idx.items()}
NUM_CLASSES = len(activity_ids)

X = data[FEATURE_NAMES].values.astype(np.float32)
y = data.activity_id.map(id2idx).values.astype(np.int32)

def make_windows(X, y, win=WIN, step=STEP):
    Xw, yw = [], []
    for i in range(0, len(X) - win + 1, step):
        xs = X[i:i+win]
        ys = y[i:i+win]
        lab = np.bincount(ys).argmax()
        Xw.append(xs)
        yw.append(lab)
    return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.int32)

Xw, yw = make_windows(X, y)
print("Windows:", Xw.shape, "Labels:", yw.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    Xw, yw, test_size=0.2, random_state=SEED, stratify=yw
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_s  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

np.save(os.path.join(OUT_DIR, "X_train_s.npy"), X_train_s)
np.save(os.path.join(OUT_DIR, "X_test_s.npy"),  X_test_s)
np.save(os.path.join(OUT_DIR, "y_train.npy"),   y_train)
np.save(os.path.join(OUT_DIR, "y_test.npy"),    y_test)

with open(os.path.join(OUT_DIR, "labels.json"), "w") as f:
    json.dump({
        "activity_ids": [int(a) for a in activity_ids],
        "idx2id": {str(k): int(v) for k,v in idx2id.items()}
    }, f, indent=2)

print("Saved arrays + labels.json")


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from keras import ops  # Keras 3 safe ops (instead of tf.*)

def encoder(x, d_model, num_heads, mlp_dim, dropout):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

    ff = layers.Dense(mlp_dim, activation="gelu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    ff = layers.Dropout(dropout)(ff)

    return layers.LayerNormalization(epsilon=1e-6)(x + ff)

def build_transformer(input_shape, num_classes,
                      patch_len=10, d_model=96, depth=3,
                      num_heads=4, mlp_dim=192,
                      dropout=0.15, lr=1e-3):

    inp = keras.Input(shape=input_shape)  # (WIN, n_features)

    # Patch embedding
    x = layers.Conv1D(
        filters=d_model,
        kernel_size=patch_len,
        strides=patch_len,
        padding="valid"
    )(inp)

    # ✅ Keras 3 safe positional embedding using keras.ops
    # seq_len = ops.shape(x)[1]  -> dynamic length (symbolic OK)
    seq_len = ops.shape(x)[1]
    pos = ops.arange(0, seq_len, dtype="int32")              # (seq_len,)
    pos_emb = layers.Embedding(input_dim=2048, output_dim=d_model)(pos)  # (seq_len, d_model)
    pos_emb = ops.expand_dims(pos_emb, axis=0)               # (1, seq_len, d_model)
    x = x + pos_emb                                         # broadcast over batch

    x = layers.Dropout(dropout)(x)

    # Encoder stack
    for _ in range(depth):
        x = encoder(x, d_model, num_heads, mlp_dim, dropout)

    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---- Fixed hyperparameters (edit here only) ----
HP = dict(
    patch_len=10,
    d_model=96,
    depth=3,
    num_heads=4,
    mlp_dim=192,
    dropout=0.15,
    lr=1e-3,
)

best_model = build_transformer(
    input_shape=(WIN, X_train_s.shape[-1]),
    num_classes=NUM_CLASSES,
    **HP
)

best_model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-5),
]

history = best_model.fit(
    X_train_s, y_train,
    validation_split=0.15,
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

keras_path = os.path.join(DIRS["models"], "transformer_best_fp32.keras")
best_model.save(keras_path)
print("Saved:", keras_path)

In [ ]:
CLASS_NAMES = [f"act_{a}" for a in activity_ids]

def plot_cm_values(cm, class_names, title, save_path, normalize=True):
    cm_plot = cm.astype(np.float32)
    if normalize:
        cm_plot = cm_plot / np.maximum(cm_plot.sum(axis=1, keepdims=True), 1e-9)

    plt.figure(figsize=(10, 9))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title, fontsize=14)
    plt.colorbar(fraction=0.046, pad=0.04)

    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right", fontsize=9)
    plt.yticks(ticks, class_names, fontsize=9)

    fmt = ".2f" if normalize else "d"
    thresh = cm_plot.max() * 0.55

    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            v = cm_plot[i, j]
            plt.text(j, i, format(v, fmt),
                     ha="center", va="center",
                     fontsize=9, fontweight="bold",
                     color="white" if v > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close()

probs = best_model.predict(X_test_s, batch_size=128, verbose=0)
y_pred = np.argmax(probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
cm_path = os.path.join(DIRS["conf_mats"], "TF_TRANSFORMER_FP32_cm_norm.png")
plot_cm_values(cm, CLASS_NAMES, "TF Transformer FP32 (Normalized CM)", cm_path, normalize=True)

report = classification_report(y_test, y_pred, target_names=CLASS_NAMES)
with open(os.path.join(DIRS["reports"], "TF_TRANSFORMER_FP32_report.txt"), "w") as f:
    f.write(report)

print("Saved CM + report")
print(report[:800])
